# 04 · False positives — is that dip really a planet?

Notebooks 00–03 and the capstone taught you to **find and confirm a signal you already know is real**. This one teaches the opposite, harder skill: **deciding whether a dip you found is actually a planet.**

This matters more than anything else in the repo. In real surveys, *most* transit-like signals are **not** planets. Kepler flagged thousands of "threshold-crossing events"; a large fraction turned out to be impostors. If you skip this step, you will "discover" a planet every week and every one will be wrong.

This is the **last teaching notebook**. After this, the answer key disappears and vetting becomes original work.

**You'll learn** the main impostors and the standard checks that expose them:
- **Eclipsing binaries** (V-shape, secondary eclipse, impossible depth)
- The **odd–even test** (the classic period-halving trap)
- **Contamination / blends** (ties back to notebook 02's aperture)
- **Stellar variability & systematics** (not box-shaped, or shared across many stars)

We'll build controlled fakes so the diagnostics are unambiguous, then watch a *real* planet (Kepler-8 b) pass the gauntlet.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(7)

def event(t, period, t0, duration, depth, ingress_frac=0.1):
    """Trapezoidal dip. ingress_frac=0.5 gives a V-shape; small gives a flat-bottomed U."""
    phase = ((t - t0 + 0.5*period) % period) - 0.5*period
    x = np.abs(phase); half = duration/2; ramp = ingress_frac*duration
    f = np.ones_like(t)
    f[x < (half-ramp)] = 1-depth
    edge = (x >= (half-ramp)) & (x < half)
    f[edge] = 1 - depth*(half-x[edge])/ramp
    return f

def fold_phase(t, period, t0):
    return (((t - t0 + 0.5*period) % period) - 0.5*period) / period  # in [-0.5, 0.5)

## 1. Eclipsing binaries: the #1 impostor

An **eclipsing binary (EB)** is two stars orbiting each other; when one passes in front of the other we see a deep dip — exactly like a transit, but caused by a star, not a planet. EBs are *far* more common than transiting planets, so they dominate any list of candidates.

Three tells give them away. Let's simulate a planet and an EB side by side and look for each.

**Tell #1 — impossible depth.** Depth ≈ (Rₚ/Rₛ)². A planet caps out around Jupiter-size, so depths above ~2–3% imply a companion *bigger than any planet* — i.e. another star. Our fake EB has a 20% eclipse: `sqrt(0.20) = 0.45`, an object 45% the size of the star. No planet does that.

**Tell #2 — V-shape.** A planet is tiny, so it sits fully on the star's disk for most of the event → flat-bottomed **U**. Two similar-sized stars are never fully overlapped → pointed **V**.

In [ ]:
t = np.arange(0, 40, 0.5/24)      # 40 days, 30-min cadence
P = 3.0

planet = event(t, P, 1.0, 0.12, 0.01, ingress_frac=0.08)                     # flat-bottomed U, 1%
# EB: deep primary (V-shape) + a shallower SECONDARY eclipse half a period later
eb = event(t, P, 1.0, 0.16, 0.20, ingress_frac=0.5) \
   * event(t, P, 1.0 + P/2, 0.16, 0.05, ingress_frac=0.5)

planet_obs = planet + rng.normal(0, 3e-4, t.size)
eb_obs     = eb     + rng.normal(0, 3e-4, t.size)

fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for a, (ph, fx, title) in zip(ax, [
        (fold_phase(t, P, 1.0), planet_obs, 'PLANET: shallow, flat-bottomed U, no secondary'),
        (fold_phase(t, P, 1.0), eb_obs,     'ECLIPSING BINARY: deep V + secondary at phase 0.5')]):
    o = np.argsort(ph); a.plot(ph[o], fx[o], '.', ms=2)
    a.axvline(0.5-1, color='r', ls=':'); a.axvline(0.5, color='r', ls=':', label='phase 0.5')
    a.set_title(title, fontsize=10); a.set_xlabel('phase'); a.legend(loc='lower right')
ax[0].set_ylabel('normalized flux'); plt.tight_layout(); plt.show()

**Tell #3 — the secondary eclipse.** When the *smaller/cooler* star passes behind the bigger one, we get a second, usually shallower dip at **phase 0.5**. Planets emit almost no light, so a real planet shows no meaningful secondary. Seeing a dip halfway between transits is a near-certain EB flag. Let's measure it rather than eyeball it:

In [ ]:
def depth_near(phase, flux, center, halfwidth=0.02):
    """Median dip (relative to the out-of-eclipse baseline) in a phase window."""
    baseline = np.median(flux[np.abs(np.abs(phase) - 0.25) < 0.05])  # a quiet stretch
    m = np.abs(phase - center) < halfwidth
    return baseline - np.median(flux[m])

ph = fold_phase(t, P, 1.0)
for name, fx in [('planet', planet_obs), ('EB', eb_obs)]:
    sec = depth_near(ph, fx, center=0.5)
    print(f'{name:6s}  secondary-eclipse depth at phase 0.5: {sec*1e6:8.0f} ppm')
print('\n-> A clear dip at phase 0.5 means starlight is being blocked there = a companion STAR, not a planet.')

## 2. The odd–even test (the period-halving trap)

Here's a subtle, extremely common mistake. If an EB's two eclipses (primary and secondary) happen to be **similar depths**, a period-finder like BLS often locks onto **half the true period** — treating the alternating primary/secondary dips as one repeating event. The signal looks beautifully periodic and you'd swear it's a planet.

The fix is the **odd–even test**: number the transits 0, 1, 2, 3… and compare the average depth of the **even** ones against the **odd** ones. For a real planet every transit is the same event → equal depths. For a mis-folded EB, evens are primaries and odds are secondaries → **different depths**. A mismatch is a red flag.

Let's fake exactly this: an EB with true period 6 d whose primary (2%) and secondary (1.5%) are close enough to be mistaken for a 3 d planet.

In [ ]:
P_true = 6.0
P_folded = 3.0     # what BLS would report (half the true period)
t0 = 1.0

# alternating deep/shallow eclipses spaced by P_true/2
ebx = event(t, P_true, t0, 0.12, 0.020, 0.15) * event(t, P_true, t0 + P_true/2, 0.12, 0.015, 0.15)
ebx_obs = ebx + rng.normal(0, 2e-4, t.size)

def odd_even_depths(t, flux, period, t0):
    n = np.round((t - t0) / period).astype(int)      # transit number for each point
    ph = fold_phase(t, period, t0)
    intransit = np.abs(ph) < 0.02
    base = np.median(flux[np.abs(np.abs(ph) - 0.25) < 0.05])
    even = base - np.median(flux[intransit & (n % 2 == 0)])
    odd  = base - np.median(flux[intransit & (n % 2 == 1)])
    return even, odd

ev, od = odd_even_depths(t, ebx_obs, P_folded, t0)
print(f'Folded at the (wrong) {P_folded} d period BLS would report:')
print(f'  even-transit depth: {ev*1e6:6.0f} ppm')
print(f'  odd-transit  depth: {od*1e6:6.0f} ppm')
print(f'  mismatch: {abs(ev-od)/max(ev,od)*100:4.0f}%  -> odd and even differ => NOT a planet (it is an EB at 2x this period)')

Run the same test on our clean planet from section 1 and the depths match — that's what "passes" looks like:

In [ ]:
ev, od = odd_even_depths(t, planet_obs, P, 1.0)
print(f'planet  even: {ev*1e6:.0f} ppm   odd: {od*1e6:.0f} ppm   mismatch: {abs(ev-od)/max(ev,od)*100:.0f}%  -> consistent = good')

## 3. Contamination and blends

This one ties straight back to **notebook 02**. Your aperture sums a patch of pixels; if another star's light falls in that patch, it **dilutes** your target. Two ways this bites:

- **Diluted depth → wrong planet size.** A deep eclipse on a faint background star, blended with a bright foreground star, can look like a *shallow* planet-sized transit on the bright star. The depth you measure is not the depth that physically happened.
- **The signal isn't even on your star.** The variable source might be a completely different star that happens to sit inside your aperture.

The professional tell is a **centroid shift**: if the image's center-of-light *moves* during the dip, the source of the dimming is off-target. Here we just show the dilution effect numerically — a true 4% eclipse, diluted by a 3× brighter neighbor in the aperture, masquerades as a ~1% "planet":

In [ ]:
true_depth = 0.04
neighbor_brightness = 3.0    # neighbor is 3x brighter than our target, and sits in the aperture
# measured depth = (light removed) / (total light in aperture)
diluted = true_depth * 1.0 / (1.0 + neighbor_brightness)
print(f'true eclipse depth on the faint star : {true_depth*1e6:.0f} ppm  (sqrt -> Rp/Rs = {np.sqrt(true_depth):.2f}, a STAR)')
print(f'measured depth after dilution        : {diluted*1e6:.0f} ppm  (sqrt -> Rp/Rs = {np.sqrt(diluted):.2f}, looks planetary!)')
print('\n-> Depth depends on WHAT ELSE is in your aperture. Always ask which pixels made the number (notebook 02),')
print('   and check whether the light centroid moves during the dip.')

## 4. Stellar variability and systematics

Not every periodic signal is an eclipse at all:

- **Starspots & pulsations** produce *smooth, sinusoidal* brightness changes as the star rotates or breathes. The tell: they're **not box-shaped**, so a **Lomb–Scargle** periodogram lights up while **BLS** (which looks for flat-bottomed boxes) stays comparatively weak. (Notebook 03 introduced both.) Rounded, continuous variation ≠ transit.
- **Instrument systematics** — thermal drifts, pointing jitter, momentum-wheel resets — create trends and jumps. The giveaway: they appear at the **same times across many unrelated stars** on the detector. A "signal" shared by your target's neighbors is the instrument, not the sky.

Rule of thumb: a transit is *brief, flat-bottomed, and periodic*. Smooth wobble → variability. Shared-with-neighbors → systematics.

## 5. A real planet passes the gauntlet: Kepler-8 b

Let's apply the two headline numeric checks — **secondary eclipse** and **odd–even** — to real data for a *bona fide* planet and confirm it comes out clean. (Same target and download as the capstone.)

In [ ]:
import lightkurve as lk

P_k8, t0_k8 = 3.52238, 131.6930    # recovered in notebook 03 / the capstone
lc = lk.search_lightcurve('KIC 6922244', author='Kepler', cadence='long')[1:5].download_all().stitch().remove_nans()
flat = lc.flatten(window_length=901)

tt = flat.time.value; ff = flat.flux.value
ph = fold_phase(tt, P_k8, t0_k8)

secondary = depth_near(ph, ff, center=0.5)
ev, od = odd_even_depths(tt, ff, P_k8, t0_k8)
transit = depth_near(ph, ff, center=0.0)

print(f'transit depth (phase 0)     : {transit*1e6:6.0f} ppm')
print(f'secondary depth (phase 0.5) : {secondary*1e6:6.0f} ppm   <- tiny vs transit => no stellar companion')
print(f'odd/even depths             : {ev*1e6:.0f} / {od*1e6:.0f} ppm   ({abs(ev-od)/max(ev,od)*100:.0f}% mismatch) => consistent')

In [ ]:
folded = flat.fold(period=P_k8, epoch_time=t0_k8)
ax = folded.scatter(s=1)
ax.axvline(P_k8/2, color='r', ls=':', label='phase 0.5 (secondary check)')
ax.axvline(-P_k8/2, color='r', ls=':')
ax.legend(); plt.show()

No secondary eclipse, matched odd/even depths, a flat-bottomed transit at a believable depth — Kepler-8 b clears every check. *That's* what earns the word "planet."

## The vetting checklist

When BLS hands you a candidate, before you get excited, run down this list:

| Check | Planet | Impostor |
|---|---|---|
| **Depth** → implied Rₚ/Rₛ | small (< ~0.2) | > ~0.3 ⇒ a star |
| **Transit shape** | flat-bottomed U | pointed V ⇒ EB |
| **Secondary eclipse** (phase 0.5) | absent | present ⇒ EB |
| **Odd–even depths** | equal | differ ⇒ EB at 2× the period |
| **Centroid** during dip | steady | shifts ⇒ blend / off-target |
| **Shape of variability** | brief box | smooth sine ⇒ spots/pulsation |
| **Shared with neighbors?** | no | yes ⇒ instrument systematic |

**The honest part.** These checks *reject* impostors; they don't *prove* a planet. Professional confirmation goes further — modeling stellar parameters, statistical validation tools (e.g. `vespa`), and often independent follow-up (radial velocity, higher-resolution imaging). As a hobbyist you usually can't get follow-up, so the right posture is: **a signal that survives every check is a promising *candidate*, not a confirmed planet.** Cross-match it against the [NASA Exoplanet Archive](https://exoplanetarchive.ipac.caltech.edu/) and the [Kepler Eclipsing Binary Catalog](http://keplerebs.villanova.edu/) — most "finds" are already known. The value is in the disciplined process, not in claiming a discovery.

You now have the full toolkit: find a signal (00–03, capstone) **and** interrogate it (here). Everything past this point is real exploration.

## Learning resources
- 🌍 [Eclipsing binary star](https://en.wikipedia.org/wiki/Binary_star#Eclipsing_binaries) — the dominant false positive
- 🗂️ [Kepler Eclipsing Binary Catalog](http://keplerebs.villanova.edu/) — check whether your "planet" is a known EB
- 🗂️ [NASA Exoplanet Archive](https://exoplanetarchive.ipac.caltech.edu/) — confirmed planets & candidate dispositions
- 📗 [Lightkurve: how to vet a transit signal](https://docs.lightkurve.org/tutorials/3-science-examples/exoplanets-identifying-transiting-planet-signals.html)
- 📄 [Morton et al. (2016), *vespa* statistical false-positive validation](https://arxiv.org/abs/1503.01738)
- 📄 [Coughlin et al. (2016), Kepler false-positive vetting (odd–even, centroids, secondaries)](https://arxiv.org/abs/1512.06149)

**You've finished the learning path.** The capstone `kepler8b_transit_recovery.ipynb` ties find + vet together on a real planet.